
# Paper 4 — Notebook 08: final remaining scientific checks

This notebook closes the three analyses that were still not fully traceable to final code:

1. **Melt-pool width verification** under V0 random, V1 source-held-out, and V2 leave-one-alloy-out evaluation.
2. **Uncertainty / selective prediction / empirical conformal intervals** after the finalized physical sanitization.
3. **Analytical defect-mode analogue** using Eagar–Tsai-derived depth/width geometry plus two training-only thresholds.

The notebook is **self-contained**. In a fresh Colab runtime it checks out the exact MeltpoolNet commit used by the frozen Paper 4 benchmark:

`6d68e2acaad8d074134a7f3d843278649774d661`

and recreates the same artifact tables/splits if needed.

## Scientific rules used here

- No held-out source/alloy is used for model tuning or preprocessing.
- Width is modeled on `log10(width)` because the target spans a large range; raw-unit metrics are also saved.
- Impossible absorptivity values are changed to missing, not silently clipped.
- The 98 material-inconsistent melting-temperature values identified in Notebook 07 are changed to missing for the uncertainty model, while all rows are retained.
- The uncertainty model is explicitly a **five-member multi-seed XGBoost ensemble**.
- The conformal calculation uses the finite-sample corrected order statistic. Because the data are clustered by source study, the reported interval result is described as **empirical source-separated split-conformal coverage**, not a formal group-conformal guarantee.
- The analytical classification check is deliberately simple. It cannot predict balling and must not be presented as a complete printability model.

**Do not force these outputs to match older manuscript tables.** If a corrected number changes, the manuscript should use the corrected number.


In [ ]:

# ============================================================
# CELL 1 — PINNED, SELF-CONTAINED SETUP
# ============================================================
import os, re, json, time, math, shutil, subprocess, sys, importlib.util, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

PKGS = {
    "pandas":"pandas", "numpy":"numpy", "scikit-learn":"sklearn",
    "pyarrow":"pyarrow", "xgboost":"xgboost", "matplotlib":"matplotlib",
    "scipy":"scipy"
}
missing = [pkg for pkg, imp in PKGS.items() if importlib.util.find_spec(imp) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    r2_score, mean_absolute_error, f1_score, matthews_corrcoef,
    balanced_accuracy_score, accuracy_score
)
from xgboost import XGBRegressor, XGBClassifier
from scipy.stats import spearmanr

import matplotlib.pyplot as plt

SEED = 42
EXPECTED_COMMIT = "6d68e2acaad8d074134a7f3d843278649774d661"
ART = "/content/artifacts"
RES = "/content/results_08"
REPO = "/content/MeltpoolNet"

os.makedirs(ART, exist_ok=True)
os.makedirs(RES, exist_ok=True)

# ---------- exact source checkout ----------
def ensure_repo():
    if not os.path.exists(os.path.join(REPO, ".git")):
        if os.path.exists(REPO):
            shutil.rmtree(REPO)
        subprocess.run(
            ["git", "clone", "https://github.com/BaratiLab/MeltpoolNet.git", REPO],
            check=True
        )
    subprocess.run(["git", "-C", REPO, "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", EXPECTED_COMMIT], check=True)
    commit = subprocess.run(
        ["git", "-C", REPO, "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True
    ).stdout.strip()
    if commit != EXPECTED_COMMIT:
        raise RuntimeError(f"Wrong MeltpoolNet commit: {commit}")
    return commit

commit = ensure_repo()

required = [
    f"{ART}/reg_clean.parquet",
    f"{ART}/cls_clean.parquet",
    f"{ART}/feature_sets.json",
    f"{ART}/meta.json",
]

def artifacts_match_expected():
    if not all(os.path.exists(p) for p in required):
        return False
    try:
        with open(f"{ART}/meta.json") as f:
            m = json.load(f)
        return m.get("data_commit") == EXPECTED_COMMIT
    except Exception:
        return False

# ---------- recreate exact frozen artifacts if absent/stale ----------
if not artifacts_match_expected():
    print("Recreating frozen artifacts from pinned MeltpoolNet commit...")

    reg0 = pd.read_csv(f"{REPO}/Data/meltpoolnet_regression.csv")
    cls0 = pd.read_csv(f"{REPO}/Data/meltpoolnet_classification.csv")

    def drop_junk(df):
        junk = [c for c in df.columns if c.startswith("Unnamed:") or str(c).strip() == ""]
        if "comment" in df.columns:
            junk.append("comment")
        return df.drop(columns=junk)

    reg0 = drop_junk(reg0)
    cls0 = drop_junk(cls0)

    CLASSES0 = ["desirable", "keyhole", "LOF", "balling"]
    CLS_ID0 = {c:i for i,c in enumerate(CLASSES0)}

    cls0 = cls0[cls0["meltpool shape"].isin(CLASSES0)].copy()
    cls0["y_class"] = cls0["meltpool shape"].map(CLS_ID0).astype(int)

    def comp_cols(df):
        return [c for c in df.columns if re.search(r"\(wt\.?%\)", c)]

    reg_F1 = ["Power","Velocity","powder flowrate","layer thickness","beam D","Hatch spacing"]
    reg_F2 = reg_F1 + ["density","Cp","k","melting T","absorption coefficient","minimum absorptivity"]
    reg_F3 = reg_F2 + ["E (J/mm)","E (J/mm3)"]
    reg_F4 = reg_F3 + comp_cols(reg0)
    reg_F5 = reg_F3 + ["Material"]

    cls_F1 = ["Power","Velocity","Hatch spacing","layer thickness","beam D"]
    cls_F2 = cls_F1 + ["density","Cp","k","melting T","absorption coefficient","minimal absorptivity"]
    cls_F3 = cls_F2 + ["p/lb","p/l","p/b2","p/b","vb","vl"]
    cls_F4 = cls_F3 + comp_cols(cls0)
    cls_F5 = cls_F3 + ["Material"]

    REG_FEATURES0 = {"F1":reg_F1,"F2":reg_F2,"F3":reg_F3,"F4":reg_F4,"F5":reg_F5}
    CLS_FEATURES0 = {"F1":cls_F1,"F2":cls_F2,"F3":cls_F3,"F4":cls_F4,"F5":cls_F5}

    def coerce_numeric(df, cols):
        for c in cols:
            if c != "Material" and c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    reg0 = coerce_numeric(reg0, sorted(set(sum(REG_FEATURES0.values(), []))))
    cls0 = coerce_numeric(cls0, sorted(set(sum(CLS_FEATURES0.values(), []))))

    reg0 = reg0.reset_index(drop=True)
    cls0 = cls0.reset_index(drop=True)

    reg0["has_depth"] = reg0["depth of meltpool"].notna()
    reg0["has_width"] = reg0["width of melt pool"].notna()
    reg0["has_class"] = reg0["meltpool shape"].isin(CLASSES0)
    reg0["y_class"] = reg0["meltpool shape"].map(CLS_ID0)

    K_REG, K_CLS = 10, 5

    def reg_splits(mask_col, k=K_REG):
        mask = reg0[mask_col].values
        idx = np.where(mask)[0]
        groups = reg0.loc[idx, "paper ID"].values
        v0 = np.full(len(reg0), -1, dtype=int)
        v1 = np.full(len(reg0), -1, dtype=int)

        for f, (_, te) in enumerate(KFold(k, shuffle=True, random_state=SEED).split(idx)):
            v0[idx[te]] = f
        for f, (_, te) in enumerate(GroupKFold(n_splits=k).split(idx, groups=groups)):
            v1[idx[te]] = f
        return v0, v1

    reg0["v0_depth"], reg0["v1_depth"] = reg_splits("has_depth")
    reg0["v0_width"], reg0["v1_width"] = reg_splits("has_width")

    def lomo_materials(mask_col, min_rows=40):
        vc = reg0.loc[reg0[mask_col], "Material"].value_counts()
        return vc[vc >= min_rows].index.tolist()

    LOMO_DEPTH = lomo_materials("has_depth")
    LOMO_WIDTH = lomo_materials("has_width")

    cy = cls0["y_class"].values
    cg = cls0["paper ID"].values
    cls0["v0"] = -1
    cls0["v1"] = -1

    for f, (_, te) in enumerate(
        StratifiedKFold(K_CLS, shuffle=True, random_state=SEED).split(cls0, cy)
    ):
        cls0.loc[te, "v0"] = f

    for f, (_, te) in enumerate(
        StratifiedGroupKFold(K_CLS, shuffle=True, random_state=SEED).split(cls0, cy, groups=cg)
    ):
        cls0.loc[te, "v1"] = f

    LOMO_CLS = cls0["Material"].value_counts()
    LOMO_CLS = LOMO_CLS[LOMO_CLS >= 40].index.tolist()

    reg0.to_parquet(f"{ART}/reg_clean.parquet", index=False)
    cls0.to_parquet(f"{ART}/cls_clean.parquet", index=False)

    with open(f"{ART}/feature_sets.json", "w") as f:
        json.dump({"regression":REG_FEATURES0, "classification":CLS_FEATURES0}, f, indent=2)

    meta0 = {
        "seed": SEED,
        "source_key": "paper ID",
        "classes": CLASSES0,
        "regression": {
            "primary_target": "depth of meltpool",
            "secondary_target": "width of melt pool",
            "appendix_target": "length of melt pool",
            "k_folds": K_REG,
            "n_depth": int(reg0["has_depth"].sum()),
            "n_width": int(reg0["has_width"].sum()),
            "n_class_labels": int(reg0["has_class"].sum()),
            "n_depth_and_width": int((reg0["has_depth"] & reg0["has_width"]).sum()),
            "lomo_depth_materials": LOMO_DEPTH,
            "lomo_width_materials": LOMO_WIDTH,
        },
        "classification": {
            "k_folds": K_CLS,
            "class_counts": {c:int((cls0["y_class"] == i).sum()) for c,i in CLS_ID0.items()},
            "lomo_materials": LOMO_CLS,
            "note": "Use pooled out-of-fold classification metrics."
        },
        "data_commit": commit,
    }
    with open(f"{ART}/meta.json", "w") as f:
        json.dump(meta0, f, indent=2)

reg = pd.read_parquet(f"{ART}/reg_clean.parquet")
cls = pd.read_parquet(f"{ART}/cls_clean.parquet")
with open(f"{ART}/feature_sets.json") as f:
    FS = json.load(f)
with open(f"{ART}/meta.json") as f:
    META = json.load(f)

if META["data_commit"] != EXPECTED_COMMIT:
    raise RuntimeError("Artifact commit mismatch.")

F3_REG = FS["regression"]["F3"]
F3_CLS = FS["classification"]["F3"]
CLASSES = META["classes"]
CLASS_IDS = np.arange(len(CLASSES))

print("Pinned Paper 4 data ready")
print("  commit                 :", META["data_commit"])
print("  regression rows        :", len(reg))
print("  width-labelled rows    :", int(reg["has_width"].sum()))
print("  depth-labelled rows    :", int(reg["has_depth"].sum()))
print("  classification rows    :", len(cls))
print("  output directory       :", RES)


In [ ]:

# ============================================================
# CELL 2 — WIDTH VERIFICATION
# ============================================================
# Nested model selection is performed within each outer training partition.
# V1 and V2 inner tuning are grouped by source study.
# Main model target: log10(width in µm).
# Raw-unit R2 / MAE are also saved to Supplement-style output.

w = reg[reg["has_width"]].reset_index(drop=True).copy()
w = w[pd.to_numeric(w["width of melt pool"], errors="coerce") > 0].reset_index(drop=True)

if len(w) != META["regression"]["n_width"]:
    print("WARNING: positive-width row count differs from frozen n_width:", len(w))

XW = w[F3_REG].copy()
YW_RAW = pd.to_numeric(w["width of melt pool"], errors="coerce").values.astype(float)
YW_LOG = np.log10(YW_RAW)
GW = w["paper ID"].astype(str).values

RIDGE_GRID = [{"alpha":a} for a in [0.1, 1.0, 10.0, 100.0]]
RF_GRID = [
    {"max_depth":md, "min_samples_leaf":leaf}
    for md in [None, 10] for leaf in [1, 3]
]
SVR_GRID = [
    {"C":C, "gamma":g, "epsilon":0.05}
    for C in [1.0, 10.0, 100.0] for g in ["scale", 0.1]
]
XGB_GRID = [
    {"max_depth":md, "learning_rate":lr, "subsample":0.8}
    for md in [3, 6] for lr in [0.05, 0.10]
]

def make_width_model(name, p):
    if name == "Ridge":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("m", Ridge(alpha=p["alpha"]))
        ])
    if name == "RF":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", RandomForestRegressor(
                n_estimators=300, random_state=SEED, n_jobs=-1,
                max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"]
            ))
        ])
    if name == "SVR":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("m", SVR(C=p["C"], gamma=p["gamma"], epsilon=p["epsilon"]))
        ])
    if name == "XGB":
        return XGBRegressor(
            n_estimators=400, random_state=SEED, n_jobs=-1, verbosity=0,
            max_depth=p["max_depth"], learning_rate=p["learning_rate"],
            subsample=p["subsample"]
        )
    raise ValueError(name)

GRIDS = {"Ridge":RIDGE_GRID, "RF":RF_GRID, "SVR":SVR_GRID, "XGB":XGB_GRID}

def inner_splits(n, groups=None):
    if groups is not None and len(np.unique(groups)) >= 3:
        return list(GroupKFold(n_splits=3).split(np.arange(n), groups=groups))
    return list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(np.arange(n)))

def tune_width(name, Xtr, ytr, groups=None):
    splits = inner_splits(len(ytr), groups)
    best_mae = np.inf
    best_p = None
    for p in GRIDS[name]:
        oof = np.full(len(ytr), np.nan)
        for itr, ite in splits:
            m = make_width_model(name, p)
            m.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = m.predict(Xtr.iloc[ite])
        mae = mean_absolute_error(ytr, oof)
        if mae < best_mae:
            best_mae, best_p = float(mae), p.copy()
    return best_p, best_mae

# Frozen V0/V1 labels follow w's original ordering because w is the has_width subset reset in order.
V0 = w["v0_width"].astype(int).values
V1 = w["v1_width"].astype(int).values

protocol_defs = {
    "V0_random": {
        "kind":"fold",
        "fold_ids":V0,
        "group_inner":False
    },
    "V1_by_study": {
        "kind":"fold",
        "fold_ids":V1,
        "group_inner":True
    },
    "V2_by_alloy": {
        "kind":"lomo",
        "materials":META["regression"]["lomo_width_materials"],
        "group_inner":True
    },
}

width_summary = []
width_tuning = []
width_preds = []

for pname, spec in protocol_defs.items():
    for model_name in ["Ridge","RF","SVR","XGB"]:
        pred_log = np.full(len(w), np.nan)
        eval_mask = np.zeros(len(w), dtype=bool)

        if spec["kind"] == "fold":
            folds = sorted(int(f) for f in np.unique(spec["fold_ids"]) if f >= 0)
            outer_iter = []
            for f in folds:
                te = spec["fold_ids"] == f
                tr = (spec["fold_ids"] != f) & (spec["fold_ids"] >= 0)
                outer_iter.append((str(f), tr, te))
        else:
            outer_iter = []
            for mat in spec["materials"]:
                te = w["Material"].astype(str).values == str(mat)
                tr = ~te
                outer_iter.append((str(mat), tr, te))

        print(f"\n{pname} — {model_name}")
        for outer_name, tr, te in outer_iter:
            if te.sum() == 0:
                continue

            Xtr = XW.loc[tr].reset_index(drop=True)
            ytr = YW_LOG[tr]
            gtr = GW[tr] if spec["group_inner"] else None

            best_p, inner_mae = tune_width(model_name, Xtr, ytr, groups=gtr)
            mdl = make_width_model(model_name, best_p)
            mdl.fit(Xtr, ytr)
            pred_log[te] = mdl.predict(XW.loc[te])
            eval_mask[te] = True

            width_tuning.append({
                "protocol":pname, "model":model_name, "outer":outer_name,
                "n_train":int(tr.sum()), "n_test":int(te.sum()),
                "inner_MAE_log":inner_mae, "best_params":json.dumps(best_p)
            })

        ok = eval_mask & np.isfinite(pred_log)
        pred_raw = 10 ** pred_log[ok]
        true_raw = YW_RAW[ok]
        rel = np.abs(pred_raw - true_raw) / true_raw * 100.0

        row = {
            "protocol":pname, "model":model_name, "n":int(ok.sum()),
            "n_studies":int(pd.Series(GW[ok]).nunique()),
            "R2_log":float(r2_score(YW_LOG[ok], pred_log[ok])),
            "MAE_log":float(mean_absolute_error(YW_LOG[ok], pred_log[ok])),
            "median_pct_error":float(np.median(rel)),
            "MAE_raw_um":float(mean_absolute_error(true_raw, pred_raw)),
            "R2_raw":float(r2_score(true_raw, pred_raw)),
        }
        width_summary.append(row)
        print(pd.Series(row).to_string())

        pf = w.loc[ok, ["paper ID","Material","width of melt pool"]].copy()
        pf["protocol"] = pname
        pf["model"] = model_name
        pf["true_log10_width"] = YW_LOG[ok]
        pf["pred_log10_width"] = pred_log[ok]
        pf["pred_width_um"] = pred_raw
        pf["pct_error"] = rel
        width_preds.append(pf)

width_summary_df = pd.DataFrame(width_summary)
width_tuning_df = pd.DataFrame(width_tuning)
width_pred_df = pd.concat(width_preds, ignore_index=True)

width_summary_df.to_csv(f"{RES}/width_verification_summary.csv", index=False)
width_tuning_df.to_csv(f"{RES}/width_nested_tuning.csv", index=False)
width_pred_df.to_csv(f"{RES}/width_oof_predictions.csv", index=False)

print("\nFINAL WIDTH TABLE")
pivot = width_summary_df.pivot(index="model", columns="protocol", values="R2_log")
print(pivot.round(3).to_string())

# Separate figure 1: log-R2 comparison
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111)
models = ["Ridge","RF","SVR","XGB"]
x = np.arange(len(models))
width_bar = 0.25
for j, p in enumerate(["V0_random","V1_by_study","V2_by_alloy"]):
    vals = [
        width_summary_df[
            (width_summary_df.model.eq(m)) & (width_summary_df.protocol.eq(p))
        ]["R2_log"].iloc[0]
        for m in models
    ]
    ax.bar(x + (j-1)*width_bar, vals, width_bar, label=p)
ax.axhline(0, linewidth=0.8, linestyle=":")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel("R² on log₁₀(width)")
ax.set_title("Melt-pool width: protocol-dependent performance")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RES}/fig_width_protocols.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_width_protocols.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# CELL 3 — FINALIZED UNCERTAINTY + EMPIRICAL CONFORMAL ANALYSIS
# ============================================================
# Exact uncertainty model identity:
#   five XGBoost regressors, random_state = 0,1,2,3,4.
#
# Final sanitization:
#   absorptivity outside (0,1] -> NaN
#   the 98 audited material-inconsistent melting-T values -> NaN
#
# Rows are retained. We do not use the uncertainty test rows for calibration/tuning.

def melting_temperature_bad_mask(df):
    T = pd.to_numeric(df["melting T"], errors="coerce")
    M = df["Material"].astype(str)
    bad = np.isclose(T, 273.0, atol=1e-8, equal_nan=False)
    bad |= M.eq("AlSi10Mg") & np.isclose(T, 1142.0, atol=1e-8)
    bad |= M.eq("HCP Cu")   & np.isclose(T, 1631.0, atol=1e-8)
    bad |= M.eq("Invar36")  & np.isclose(T, 2000.0, atol=1e-8)
    bad |= M.eq("MS1-")     & np.isclose(T, 2848.0, atol=1e-8)
    return pd.Series(bad, index=df.index)

def sanitize_depth_table(df):
    d = df.copy()
    a = pd.to_numeric(d["absorption coefficient"], errors="coerce")
    d["absorption coefficient"] = np.where((a > 0) & (a <= 1), a, np.nan)
    bad_tm = melting_temperature_bad_mask(d)
    d.loc[bad_tm, "melting T"] = np.nan
    return d

def xgb_unc(seed):
    return XGBRegressor(
        random_state=seed,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        verbosity=0,
        n_jobs=-1
    )

def finite_sample_q(abs_resid, alpha=0.10):
    r = np.sort(np.asarray(abs_resid, dtype=float))
    r = r[np.isfinite(r)]
    n = len(r)
    if n == 0:
        return np.nan
    # Split-conformal corrected order statistic:
    # k = ceil((n+1)*(1-alpha)), clipped to n.
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(r[k - 1])

def make_group_folds(groups, n_splits=10):
    n = min(n_splits, len(np.unique(groups)))
    fold = np.full(len(groups), -1, dtype=int)
    for f, (_, te) in enumerate(
        GroupKFold(n_splits=n).split(np.arange(len(groups)), groups=groups)
    ):
        fold[te] = f
    return fold

def run_uncertainty(scope_name, d, fold_ids):
    d = d.reset_index(drop=True).copy()
    X = d[F3_REG].copy()
    y_raw = pd.to_numeric(d["depth of meltpool"], errors="coerce").values.astype(float)
    y = np.log10(y_raw)
    groups = d["paper ID"].astype(str).values

    pred_mean = np.full(len(d), np.nan)
    pred_sd = np.full(len(d), np.nan)
    knn_dist = np.full(len(d), np.nan)
    q_fold = np.full(len(d), np.nan)
    cal_n_fold = np.full(len(d), np.nan)
    cal_groups_fold = np.full(len(d), np.nan)

    fold_rows = []
    rng_master = np.random.default_rng(SEED)

    for f in sorted(int(v) for v in np.unique(fold_ids) if v >= 0):
        te = fold_ids == f
        train_outer = (fold_ids != f) & (fold_ids >= 0)

        train_group_ids = np.unique(groups[train_outer])
        if len(train_group_ids) < 4:
            raise RuntimeError(f"Fold {f}: too few training studies for calibration split.")

        # Deterministic 25% of TRAINING STUDIES reserved for calibration.
        rng = np.random.default_rng(SEED + 1000 + f)
        shuffled = train_group_ids.copy()
        rng.shuffle(shuffled)
        n_cal_groups = max(1, int(np.ceil(0.25 * len(shuffled))))
        cal_group_ids = set(shuffled[:n_cal_groups])

        cal = train_outer & np.array([g in cal_group_ids for g in groups])
        fit = train_outer & (~cal)

        if len(np.unique(groups[fit])) < 2:
            raise RuntimeError(f"Fold {f}: training-study split failed.")

        models = []
        cal_members = []
        test_members = []

        for seed in range(5):
            m = xgb_unc(seed)
            m.fit(X.loc[fit], y[fit])
            models.append(m)
            cal_members.append(m.predict(X.loc[cal]))
            test_members.append(m.predict(X.loc[te]))

        cal_members = np.vstack(cal_members)
        test_members = np.vstack(test_members)
        cal_mu = cal_members.mean(axis=0)
        test_mu = test_members.mean(axis=0)
        test_sd = test_members.std(axis=0)

        resid = np.abs(y[cal] - cal_mu)
        q = finite_sample_q(resid, alpha=0.10)

        pred_mean[te] = test_mu
        pred_sd[te] = test_sd
        q_fold[te] = q
        cal_n_fold[te] = int(cal.sum())
        cal_groups_fold[te] = int(len(cal_group_ids))

        # Training-support distance from the model-fitting partition only.
        imp = SimpleImputer(strategy="median").fit(X.loc[fit])
        Xfit_imp = imp.transform(X.loc[fit])
        Xte_imp = imp.transform(X.loc[te])
        sc = StandardScaler().fit(Xfit_imp)
        Xfit_std = sc.transform(Xfit_imp)
        Xte_std = sc.transform(Xte_imp)

        k_nn = min(5, len(Xfit_std))
        nn = NearestNeighbors(n_neighbors=k_nn).fit(Xfit_std)
        dist, _ = nn.kneighbors(Xte_std)
        knn_dist[te] = dist.mean(axis=1)

        fold_rows.append({
            "scope":scope_name, "fold":f,
            "n_test":int(te.sum()),
            "n_outer_train":int(train_outer.sum()),
            "n_model_fit":int(fit.sum()),
            "n_calibration":int(cal.sum()),
            "n_fit_studies":int(pd.Series(groups[fit]).nunique()),
            "n_cal_studies":int(len(cal_group_ids)),
            "conformal_q_log":q,
        })

    ok = (
        np.isfinite(pred_mean) & np.isfinite(pred_sd) &
        np.isfinite(knn_dist) & np.isfinite(q_fold)
    )

    true_log = y[ok]
    mu = pred_mean[ok]
    sd = pred_sd[ok]
    dist = knn_dist[ok]
    q = q_fold[ok]
    true_raw = y_raw[ok]
    pred_raw = 10 ** mu
    pct = np.abs(pred_raw - true_raw) / true_raw * 100.0
    abs_log_err = np.abs(mu - true_log)
    covered = (true_log >= mu - q) & (true_log <= mu + q)

    rho_sd = spearmanr(abs_log_err, sd, nan_policy="omit").statistic
    rho_dist = spearmanr(abs_log_err, dist, nan_policy="omit").statistic

    order_sd = np.argsort(sd)
    order_dist = np.argsort(dist)

    selective_rows = []
    for frac in [0.10, 0.25, 0.50, 0.75, 1.00]:
        n_keep = max(1, int(np.ceil(frac * len(true_log))))
        for ranker, order in [("ensemble_disagreement",order_sd), ("training_distance",order_dist)]:
            idx = order[:n_keep]
            selective_rows.append({
                "scope":scope_name,
                "ranker":ranker,
                "retained_fraction":frac,
                "n":int(n_keep),
                "median_pct_error":float(np.median(pct[idx])),
                "R2_log":float(r2_score(true_log[idx], mu[idx])) if n_keep > 1 else np.nan,
            })

    summary = {
        "scope":scope_name,
        "n":int(len(true_log)),
        "n_studies":int(pd.Series(groups[ok]).nunique()),
        "R2_log":float(r2_score(true_log, mu)),
        "median_pct_error_all":float(np.median(pct)),
        "median_pct_error_top25_by_disagreement":float(
            np.median(pct[order_sd[:max(1, int(np.ceil(0.25*len(pct))))]])
        ),
        "empirical_conformal90_coverage":float(np.mean(covered)),
        "median_conformal_halfwidth_log10":float(np.median(q)),
        "spearman_abslogerr_vs_ensemble_sd":float(rho_sd),
        "spearman_abslogerr_vs_knn_distance":float(rho_dist),
    }

    pred_out = d.loc[ok, ["paper ID","Material","Process","Sub-process","depth of meltpool"]].copy()
    pred_out["scope"] = scope_name
    pred_out["true_log10_depth"] = true_log
    pred_out["pred_log10_depth"] = mu
    pred_out["pred_depth_um"] = pred_raw
    pred_out["ensemble_sd_log10"] = sd
    pred_out["knn_distance"] = dist
    pred_out["conformal_q_log10"] = q
    pred_out["covered_90"] = covered
    pred_out["pct_error"] = pct

    return (
        pd.DataFrame([summary]),
        pd.DataFrame(selective_rows),
        pd.DataFrame(fold_rows),
        pred_out,
    )

# Full finalized-sanitized depth table.
d_full = sanitize_depth_table(
    reg[reg["has_depth"]].reset_index(drop=True)
)
fold_full = d_full["v1_depth"].astype(int).values

# PBF-only is rerun as its own grouped protocol, rather than filtering predictions
# from a model trained on electron-beam / DED records.
d_pbf = d_full[d_full["Process"].eq("PBF")].reset_index(drop=True)
fold_pbf = make_group_folds(d_pbf["paper ID"].astype(str).values, n_splits=10)

unc_outputs = []
sel_outputs = []
fold_outputs = []
pred_outputs = []

for name, data, folds in [
    ("all_data_final_sanitized", d_full, fold_full),
    ("PBF_only_final_sanitized", d_pbf, fold_pbf),
]:
    s, st, ft, po = run_uncertainty(name, data, folds)
    unc_outputs.append(s)
    sel_outputs.append(st)
    fold_outputs.append(ft)
    pred_outputs.append(po)

unc_summary = pd.concat(unc_outputs, ignore_index=True)
selective = pd.concat(sel_outputs, ignore_index=True)
unc_folds = pd.concat(fold_outputs, ignore_index=True)
unc_pred = pd.concat(pred_outputs, ignore_index=True)

unc_summary.to_csv(f"{RES}/uncertainty_final_summary.csv", index=False)
selective.to_csv(f"{RES}/uncertainty_selective_prediction.csv", index=False)
unc_folds.to_csv(f"{RES}/uncertainty_fold_calibration.csv", index=False)
unc_pred.to_csv(f"{RES}/uncertainty_oof_predictions.csv", index=False)

print("\nFINALIZED UNCERTAINTY SUMMARY")
print(unc_summary.round(3).to_string(index=False))

# Separate figure 2: error by disagreement decile, full dataset.
p = unc_pred[unc_pred.scope.eq("all_data_final_sanitized")].copy()
p["disagreement_decile"] = pd.qcut(
    p["ensemble_sd_log10"], q=10, labels=False, duplicates="drop"
) + 1
dec = p.groupby("disagreement_decile", as_index=False).agg(
    median_abs_log_error=("true_log10_depth", lambda x: np.nan)
)
# Explicit aggregation because lambda above cannot access prediction column.
dec = (
    p.assign(abs_log_error=np.abs(p["pred_log10_depth"] - p["true_log10_depth"]))
     .groupby("disagreement_decile", as_index=False)["abs_log_error"].median()
)

fig = plt.figure(figsize=(7, 4.5))
ax = fig.add_subplot(111)
ax.plot(dec["disagreement_decile"], dec["abs_log_error"], marker="o")
ax.set_xlabel("Ensemble-disagreement decile")
ax.set_ylabel("Median absolute log₁₀ error")
ax.set_title("Error versus five-member XGBoost disagreement")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RES}/fig_uncertainty_error_vs_disagreement.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_uncertainty_error_vs_disagreement.png", dpi=600, bbox_inches="tight")
plt.show()

# Separate figure 3: selective prediction.
sp = selective[selective.scope.eq("all_data_final_sanitized")].copy()
fig = plt.figure(figsize=(7, 4.5))
ax = fig.add_subplot(111)
for ranker in sp["ranker"].unique():
    t = sp[sp.ranker.eq(ranker)].sort_values("retained_fraction")
    ax.plot(t["retained_fraction"], t["median_pct_error"], marker="o", label=ranker)
ax.set_xlabel("Fraction of predictions retained")
ax.set_ylabel("Median relative error (%)")
ax.set_title("Selective prediction under source-held-out evaluation")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RES}/fig_uncertainty_selective_prediction.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_uncertainty_selective_prediction.png", dpi=600, bbox_inches="tight")
plt.show()

# Separate figure 4: predicted vs observed.
fig = plt.figure(figsize=(5.5, 5.0))
ax = fig.add_subplot(111)
scat = ax.scatter(
    p["true_log10_depth"], p["pred_log10_depth"],
    c=p["ensemble_sd_log10"], s=12
)
mn = min(p["true_log10_depth"].min(), p["pred_log10_depth"].min())
mx = max(p["true_log10_depth"].max(), p["pred_log10_depth"].max())
ax.plot([mn,mx], [mn,mx], linestyle="--", linewidth=1)
ax.set_xlabel("Observed log₁₀(depth µm)")
ax.set_ylabel("Predicted log₁₀(depth µm)")
ax.set_title("Source-held-out uncertainty predictions")
fig.colorbar(scat, ax=ax, label="ensemble disagreement")
plt.tight_layout()
plt.savefig(f"{RES}/fig_uncertainty_predicted_vs_observed.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_uncertainty_predicted_vs_observed.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# CELL 4 — EAGAR–TSAI-DERIVED DEFECT-MODE ANALOGUE
# ============================================================
# This is deliberately a SIMPLE comparison, not a full printability framework.
#
# Physical baseline:
#   keyhole indicator = E-T depth / E-T width  (large -> keyhole)
#   LOF indicator     = E-T depth / layer thickness (small -> LOF)
#
# Two thresholds and rule order are selected using OUTER TRAINING labels only.
# The baseline has no balling criterion, so "balling" is never predicted.
#
# Learned comparator:
#   XGBoost classifier on the same compatible rows and same outer folds,
#   with inner tuning performed only on the training partition.
#
# If the final compatible-row count differs from the older manuscript's 555,
# report the new count. Do not force 555.

NQ = 320
trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def et_dT_xyz(w, y, z, P, v, sigma, rho, cp, k, eta):
    alpha = k / (rho * cp)
    q = eta * P
    tmax = max(
        50 * sigma**2 / alpha,
        50 * sigma / max(v, 1e-12),
        1e-4
    )
    tau = np.logspace(np.log10(tmax * 1e-9), np.log10(tmax), NQ)
    a4 = 4 * alpha * tau
    s2 = a4 + 2 * sigma**2
    expo = -((w + v*tau)**2 + y**2) / s2 - (z**2) / a4
    integrand = np.where(expo < -700, 0.0, np.exp(expo))
    integrand = integrand / (s2 * np.sqrt(tau))
    val = trap(integrand, tau)
    return q / (np.pi * rho * cp * np.sqrt(4*np.pi*alpha)) * val

def root_extent(axis, wh, dTm, P, v, sigma, rho, cp, k, eta):
    lo, hi = 1e-12, sigma

    def val(ext):
        if axis == "z":
            return et_dT_xyz(wh, 0.0, ext, P, v, sigma, rho, cp, k, eta)
        return et_dT_xyz(wh, ext, 1e-12, P, v, sigma, rho, cp, k, eta)

    if val(lo) < dTm:
        return 0.0

    for _ in range(40):
        if val(hi) < dTm:
            break
        hi *= 1.6
    else:
        return np.nan

    for _ in range(55):
        mid = 0.5 * (lo + hi)
        if val(mid) >= dTm:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def et_geometry_fast(P, v, beamD, rho, cp, k, Tm, eta=0.35, T0=293.0, sigma_factor=0.25):
    vals = np.array([P,v,beamD,rho,cp,k,Tm,eta], dtype=float)
    if not np.all(np.isfinite(vals)):
        return np.nan, np.nan
    if min(P,v,beamD,rho,cp,k,eta) <= 0 or Tm <= T0:
        return np.nan, np.nan

    sigma = sigma_factor * beamD
    dTm = Tm - T0

    ws = np.linspace(-6*sigma, 2*sigma, 17)
    surf = np.array([
        et_dT_xyz(w0, 0.0, 1e-12, P, v, sigma, rho, cp, k, eta)
        for w0 in ws
    ])
    if not np.isfinite(surf).any() or np.nanmax(surf) < dTm:
        return 0.0, 0.0

    wh = ws[int(np.nanargmax(surf))]
    depth_m = root_extent("z", wh, dTm, P,v,sigma,rho,cp,k,eta)
    halfwidth_m = root_extent("y", wh, dTm, P,v,sigma,rho,cp,k,eta)
    width_m = 2.0 * halfwidth_m if np.isfinite(halfwidth_m) else np.nan
    return depth_m, width_m

def cls_tm_bad(df):
    # Same audited material-specific decisions used for Notebook 07.
    T = pd.to_numeric(df["melting T"], errors="coerce")
    M = df["Material"].astype(str)
    bad = np.isclose(T, 273.0, atol=1e-8, equal_nan=False)
    bad |= M.eq("AlSi10Mg") & np.isclose(T, 1142.0, atol=1e-8)
    bad |= M.eq("HCP Cu")   & np.isclose(T, 1631.0, atol=1e-8)
    bad |= M.eq("Invar36")  & np.isclose(T, 2000.0, atol=1e-8)
    bad |= M.eq("MS1-")     & np.isclose(T, 2848.0, atol=1e-8)
    return pd.Series(bad, index=df.index)

c = cls.reset_index(drop=True).copy()
a = pd.to_numeric(c["absorption coefficient"], errors="coerce")
eta = np.where((a > 0) & (a <= 1), a, 0.35)

beam = pd.to_numeric(c["beam D"], errors="coerce")
tm_bad = cls_tm_bad(c)

needed = ["Power","Velocity","beam D","density","Cp","k","melting T","layer thickness"]
complete = np.ones(len(c), dtype=bool)
for col in needed:
    z = pd.to_numeric(c[col], errors="coerce")
    complete &= np.isfinite(z)
    complete &= z > 0
complete &= ~tm_bad.values
complete &= ~(beam.notna().values & (beam.values < 1))

et_depth = np.full(len(c), np.nan)
et_width = np.full(len(c), np.nan)

cache_path = f"{RES}/classification_et_geometry_cache.csv"
if os.path.exists(cache_path):
    cache = pd.read_csv(cache_path)
    if len(cache) == len(c):
        et_depth = cache["ET_depth_um"].values
        et_width = cache["ET_width_um"].values
        print("Loaded E-T classification geometry cache.")
    else:
        os.remove(cache_path)

if not np.isfinite(et_depth).any():
    idx = np.where(complete)[0]
    t0 = time.time()
    for jj, i in enumerate(idx, 1):
        r = c.iloc[i]
        d_m, w_m = et_geometry_fast(
            P=float(r["Power"]),
            v=float(r["Velocity"]) / 1000.0,  # mm/s -> m/s
            beamD=float(r["beam D"]) * 1e-6, # µm -> m
            rho=float(r["density"]),
            cp=float(r["Cp"]),
            k=float(r["k"]),
            Tm=float(r["melting T"]),
            eta=float(eta[i]),
            T0=293.0,
            sigma_factor=0.25
        )
        et_depth[i] = d_m * 1e6 if np.isfinite(d_m) else np.nan
        et_width[i] = w_m * 1e6 if np.isfinite(w_m) else np.nan
        if jj % 100 == 0:
            print(f"  E-T classification geometry: {jj}/{len(idx)}")
    pd.DataFrame({
        "ET_depth_um":et_depth,
        "ET_width_um":et_width
    }).to_csv(cache_path, index=False)
    print(f"E-T geometry completed in {(time.time()-t0)/60:.1f} min")

layer = pd.to_numeric(c["layer thickness"], errors="coerce").values.astype(float)

compatible = (
    complete &
    np.isfinite(et_depth) & (et_depth > 0) &
    np.isfinite(et_width) & (et_width > 0) &
    np.isfinite(layer) & (layer > 0)
)

cc = c.loc[compatible].reset_index(drop=True)
cc["ET_depth_um"] = et_depth[compatible]
cc["ET_width_um"] = et_width[compatible]
cc["keyhole_ratio_ET_d_over_w"] = cc["ET_depth_um"] / cc["ET_width_um"]
cc["LOF_ratio_ET_d_over_layer"] = cc["ET_depth_um"] / pd.to_numeric(
    cc["layer thickness"], errors="coerce"
)

print("\nE-T classification-compatible subset")
print("  rows   :", len(cc))
print("  studies:", cc["paper ID"].nunique())
print("  classes:")
print(cc["meltpool shape"].value_counts().to_string())

cc.to_csv(f"{RES}/et_classification_compatible_rows.csv", index=False)

def physical_predict(rk, rl, t_key, t_lof, order):
    pred = np.zeros(len(rk), dtype=int)  # desirable = 0
    key = rk >= t_key
    lof = rl <= t_lof

    if order == "lof_first":
        pred[key] = 1
        pred[lof] = 2
    else:
        pred[lof] = 2
        pred[key] = 1
    # balling (3) is intentionally never predicted.
    return pred

def tune_physical_thresholds(rk, rl, y):
    # Candidate thresholds use training-data quantiles only.
    qs = np.linspace(0.05, 0.95, 25)
    kgrid = np.unique(np.quantile(rk[np.isfinite(rk)], qs))
    lgrid = np.unique(np.quantile(rl[np.isfinite(rl)], qs))

    best = (-np.inf, None)
    for order in ["keyhole_first","lof_first"]:
        for tk in kgrid:
            for tl in lgrid:
                pred = physical_predict(rk, rl, tk, tl, order)
                score = f1_score(
                    y, pred, labels=CLASS_IDS, average="macro", zero_division=0
                )
                if score > best[0]:
                    best = (score, {"t_key":float(tk), "t_lof":float(tl), "order":order})
    return best[1], float(best[0])

CLS_XGB_GRID = [
    {"max_depth":md, "learning_rate":lr, "subsample":0.8}
    for md in [3,6] for lr in [0.05,0.10]
]

def make_cls_xgb(p):
    return XGBClassifier(
        random_state=SEED,
        n_estimators=400,
        max_depth=p["max_depth"],
        learning_rate=p["learning_rate"],
        subsample=p["subsample"],
        objective="multi:softprob",
        num_class=len(CLASSES),
        eval_metric="mlogloss",
        verbosity=0,
        n_jobs=-1
    )

def tune_cls_xgb(Xtr, ytr, groups=None):
    if groups is not None and len(np.unique(groups)) >= 3:
        splits = list(StratifiedGroupKFold(
            n_splits=3, shuffle=True, random_state=SEED
        ).split(Xtr, ytr, groups=groups))
    else:
        splits = list(StratifiedKFold(
            n_splits=3, shuffle=True, random_state=SEED
        ).split(Xtr, ytr))

    best = (-np.inf, None)
    for p in CLS_XGB_GRID:
        oof = np.full(len(ytr), -1, dtype=int)
        for itr, ite in splits:
            if len(np.unique(ytr[itr])) < 2:
                continue
            m = make_cls_xgb(p)
            m.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = m.predict(Xtr.iloc[ite]).astype(int)
        ok = oof >= 0
        if ok.sum() == 0:
            continue
        score = f1_score(
            ytr[ok], oof[ok], labels=CLASS_IDS, average="macro", zero_division=0
        )
        if score > best[0]:
            best = (float(score), p.copy())
    if best[1] is None:
        raise RuntimeError("No valid learned-classifier tuning configuration.")
    return best[1], best[0]

XC = cc[F3_CLS].copy()
YC = cc["y_class"].astype(int).values
GC = cc["paper ID"].astype(str).values
RK = cc["keyhole_ratio_ET_d_over_w"].values.astype(float)
RL = cc["LOF_ratio_ET_d_over_layer"].values.astype(float)

# Reuse frozen fold labels carried into cc.
protocols = {
    "V0_random": (cc["v0"].astype(int).values, False),
    "V1_by_study": (cc["v1"].astype(int).values, True),
}

class_rows = []
class_tuning = []
class_preds = []

for pname, (fold_ids, group_inner) in protocols.items():
    phys = np.full(len(cc), -1, dtype=int)
    learned = np.full(len(cc), -1, dtype=int)

    for f in sorted(int(v) for v in np.unique(fold_ids) if v >= 0):
        te = fold_ids == f
        tr = (fold_ids != f) & (fold_ids >= 0)
        if te.sum() == 0:
            continue

        # Physical threshold calibration on OUTER TRAINING rows only.
        pthr, train_f1 = tune_physical_thresholds(RK[tr], RL[tr], YC[tr])
        phys[te] = physical_predict(
            RK[te], RL[te], pthr["t_key"], pthr["t_lof"], pthr["order"]
        )

        # Learned comparator with inner tuning only on outer training data.
        gtr = GC[tr] if group_inner else None
        best_p, inner_f1 = tune_cls_xgb(
            XC.loc[tr].reset_index(drop=True), YC[tr], groups=gtr
        )
        m = make_cls_xgb(best_p)
        m.fit(XC.loc[tr], YC[tr])
        learned[te] = m.predict(XC.loc[te]).astype(int)

        class_tuning.append({
            "protocol":pname, "outer_fold":f,
            "n_train":int(tr.sum()), "n_test":int(te.sum()),
            "physical_train_macroF1":train_f1,
            "physical_t_key":pthr["t_key"],
            "physical_t_lof":pthr["t_lof"],
            "physical_rule_order":pthr["order"],
            "learned_inner_macroF1":inner_f1,
            "learned_best_params":json.dumps(best_p),
        })

    ok = (phys >= 0) & (learned >= 0)
    f_phys = f1_score(YC[ok], phys[ok], labels=CLASS_IDS, average="macro", zero_division=0)
    f_ml = f1_score(YC[ok], learned[ok], labels=CLASS_IDS, average="macro", zero_division=0)

    row = {
        "protocol":pname,
        "n":int(ok.sum()),
        "n_studies":int(pd.Series(GC[ok]).nunique()),
        "physical_macroF1":float(f_phys),
        "learned_XGB_macroF1":float(f_ml),
        "advantage_macroF1":float(f_ml - f_phys),
        "physical_MCC":float(matthews_corrcoef(YC[ok], phys[ok])),
        "learned_XGB_MCC":float(matthews_corrcoef(YC[ok], learned[ok])),
    }
    class_rows.append(row)

    pf = cc.loc[ok, ["paper ID","Material","meltpool shape","y_class",
                     "ET_depth_um","ET_width_um",
                     "keyhole_ratio_ET_d_over_w","LOF_ratio_ET_d_over_layer"]].copy()
    pf["protocol"] = pname
    pf["physical_pred_id"] = phys[ok]
    pf["physical_pred"] = [CLASSES[i] for i in phys[ok]]
    pf["learned_pred_id"] = learned[ok]
    pf["learned_pred"] = [CLASSES[i] for i in learned[ok]]
    class_preds.append(pf)

class_summary = pd.DataFrame(class_rows)
class_tuning_df = pd.DataFrame(class_tuning)
class_pred_df = pd.concat(class_preds, ignore_index=True)

# Source-level bootstrap for advantage.
def bootstrap_adv(frame, B=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    src = frame["paper ID"].astype(str).values
    us = np.unique(src)
    src_idx = {s:np.where(src == s)[0] for s in us}
    vals = []

    yt = frame["y_class"].values.astype(int)
    pp = frame["physical_pred_id"].values.astype(int)
    mp = frame["learned_pred_id"].values.astype(int)

    for _ in range(B):
        sampled = rng.choice(us, size=len(us), replace=True)
        idx = np.concatenate([src_idx[s] for s in sampled])
        f1p = f1_score(yt[idx], pp[idx], labels=CLASS_IDS, average="macro", zero_division=0)
        f1m = f1_score(yt[idx], mp[idx], labels=CLASS_IDS, average="macro", zero_division=0)
        vals.append(f1m - f1p)

    return float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))

for i, r in class_summary.iterrows():
    frame = class_pred_df[class_pred_df.protocol.eq(r["protocol"])].reset_index(drop=True)
    lo, hi = bootstrap_adv(frame)
    class_summary.loc[i, "advantage_CI95_lo"] = lo
    class_summary.loc[i, "advantage_CI95_hi"] = hi

class_summary.to_csv(f"{RES}/et_classification_summary.csv", index=False)
class_tuning_df.to_csv(f"{RES}/et_classification_thresholds_and_tuning.csv", index=False)
class_pred_df.to_csv(f"{RES}/et_classification_oof_predictions.csv", index=False)

# Per-class F1.
perclass_rows = []
for pname in class_summary["protocol"]:
    frame = class_pred_df[class_pred_df.protocol.eq(pname)]
    yt = frame["y_class"].values.astype(int)
    for pred_name, col in [
        ("physical_two_threshold","physical_pred_id"),
        ("learned_XGB","learned_pred_id")
    ]:
        scores = f1_score(
            yt, frame[col].values.astype(int),
            labels=CLASS_IDS, average=None, zero_division=0
        )
        for cls_name, score in zip(CLASSES, scores):
            perclass_rows.append({
                "protocol":pname, "predictor":pred_name,
                "class":cls_name, "F1":float(score)
            })
pd.DataFrame(perclass_rows).to_csv(
    f"{RES}/et_classification_per_class_F1.csv", index=False
)

print("\nE-T-DERIVED CLASSIFICATION ANALOGUE")
print(class_summary.round(3).to_string(index=False))

# Separate figure 5: physical vs learned macro-F1.
fig = plt.figure(figsize=(6.5, 4.5))
ax = fig.add_subplot(111)
x = np.arange(len(class_summary))
bw = 0.34
ax.bar(x - bw/2, class_summary["physical_macroF1"], bw, label="two-threshold physical baseline")
ax.bar(x + bw/2, class_summary["learned_XGB_macroF1"], bw, label="XGBoost")
ax.set_xticks(x)
ax.set_xticklabels(["V0 random","V1 unseen study"])
ax.set_ylabel("Macro-F1")
ax.set_ylim(0,1)
ax.set_title("Defect-mode analogue on E-T-compatible records")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{RES}/fig_et_classification.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_et_classification.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# CELL 5 — MANUSCRIPT DECISION SUMMARY + PACKAGE
# ============================================================
# This file is meant to make the next manuscript edit mechanical:
# it records final numbers and explicitly identifies old claims that
# should be kept, revised, or removed.

lines = []
lines.append("PAPER 4 — NOTEBOOK 08 FINAL REMAINING CHECKS")
lines.append("="*58)
lines.append("")
lines.append(f"Data commit: {META['data_commit']}")
lines.append("")

lines.append("1) WIDTH VERIFICATION")
for _, r in width_summary_df.iterrows():
    lines.append(
        f"{r['model']} | {r['protocol']} | n={int(r['n'])} | "
        f"R2_log={r['R2_log']:.3f} | median_error={r['median_pct_error']:.1f}% | "
        f"R2_raw={r['R2_raw']:.3f}"
    )

lines.append("")
lines.append("2) UNCERTAINTY / EMPIRICAL SOURCE-SEPARATED CONFORMAL")
for _, r in unc_summary.iterrows():
    lines.append(
        f"{r['scope']} | n={int(r['n'])} | R2_log={r['R2_log']:.3f} | "
        f"median_error={r['median_pct_error_all']:.1f}% | "
        f"top25_error={r['median_pct_error_top25_by_disagreement']:.1f}% | "
        f"coverage90={r['empirical_conformal90_coverage']:.3f} | "
        f"rho(error,ensemble_sd)={r['spearman_abslogerr_vs_ensemble_sd']:.3f} | "
        f"rho(error,knn)={r['spearman_abslogerr_vs_knn_distance']:.3f}"
    )

lines.append("")
lines.append("3) E-T DEFECT-MODE ANALOGUE")
for _, r in class_summary.iterrows():
    lines.append(
        f"{r['protocol']} | n={int(r['n'])} | physical_macroF1={r['physical_macroF1']:.3f} | "
        f"XGB_macroF1={r['learned_XGB_macroF1']:.3f} | "
        f"advantage={r['advantage_macroF1']:.3f} "
        f"[{r['advantage_CI95_lo']:.3f},{r['advantage_CI95_hi']:.3f}]"
    )

lines.append("")
lines.append("MANUSCRIPT RULES")
lines.append("- Use these Notebook 08 numbers instead of older unverified width/uncertainty/classification-analogue values.")
lines.append("- Say 'five-member multi-seed XGBoost ensemble'.")
lines.append("- Say 'empirical source-separated split-conformal coverage'; do not claim a formal group-conformal guarantee.")
lines.append("- The two-threshold E-T classification analogue is deliberately incomplete and never predicts balling.")
lines.append("- If the compatible classification row count is not 555, replace the old 555 claim.")
lines.append("- Keep source study as a provenance proxy, not as a verified laboratory identifier.")

summary_path = f"{RES}/NOTEBOOK08_MANUSCRIPT_DECISIONS.txt"
with open(summary_path, "w") as f:
    f.write("\n".join(lines))

print("\n".join(lines))

archive = "/content/Paper4_08_remaining_checks_results"
shutil.make_archive(archive, "zip", RES)

print("\nCreated final ZIP:")
print(archive + ".zip")
print("\nDownload that ZIP from the Colab Files panel and upload it to ChatGPT.")
print("\nGenerated files:")
for fn in sorted(os.listdir(RES)):
    print(" ", fn)
